# MVO 正确性测试

从上到下运行：先加载实现，再逐段阅读 15 项测试，最后执行测试集。每个测试单独放在一个代码单元格；修改实现后重新运行加载单元格及测试集。测试使用小型模拟数据和手算结果，不需要下载市场数据。


In [1]:
%run ./mvo_utils.ipynb


Note: you may need to restart the kernel to use updated packages.


In [2]:
from contextlib import redirect_stderr, redirect_stdout
from io import BytesIO, StringIO
from pathlib import Path
import tempfile
import unittest
from zipfile import ZipFile

# unittest 的断言会给出具体的失败信息。
checks = unittest.TestCase()
torch.set_num_threads(1)


## 1. 未来数据不能影响当前特征

只修改配置日之后的收益；历史特征、均值与协方差因子必须保持相同，未来标签必须改变。


In [3]:
def test_future_changes_do_not_change_current_features():
        frame = synthetic_returns(days=250, assets=3)
        original = make_samples(frame)
        frame.iloc[126:] += 0.01
        modified = make_samples(frame)
        for key in ("features", "factors", "means"):
            np.testing.assert_allclose(original[key][0], modified[key][0])
        checks.assertFalse(np.allclose(original["future"][0], modified["future"][0]))


## 2. 初始现金、费用与权重漂移

使用可以手算的两日收益，核对首次买入费用、累计净值和自然漂移后的权重。


In [4]:
def test_drift_cost_and_initial_cash():
        returns = np.array([[0.1, 0.0], [0.0, 0.2]])
        daily, drifted, turnover = hold_period(np.array([0.5, 0.5]), np.zeros(2), returns, 10)
        checks.assertAlmostEqual(turnover, 1)
        checks.assertAlmostEqual(np.prod(1 + daily), 0.999 * 1.15)
        np.testing.assert_allclose(drifted, [0.55 / 1.15, 0.6 / 1.15])
        _, _, next_turnover = hold_period(np.array([0.5, 0.5]), drifted, returns, 0)
        checks.assertAlmostEqual(next_turnover, np.abs(0.5 - drifted).sum())


## 3. 训练收益和回测记账一致

关闭交易成本后，PyTorch 可微持有期收益应等于 NumPy 回测记账结果。


In [5]:
def test_differentiable_holding_matches_accounting():
        weights = np.array([0.3, 0.7])
        returns = np.array([[0.1, -0.1], [-0.05, 0.2]])
        actual = buy_and_hold_returns(torch.tensor(weights[None]), torch.tensor(returns[None]))
        expected, _, _ = hold_period(weights, weights, returns, 0)
        np.testing.assert_allclose(actual.numpy()[0], expected)


## 4. 最小方差解析解

对于不相关的两个资产，最优权重与方差的倒数成正比；这提供独立的求解器检查。


In [6]:
def test_minimum_variance_matches_analytic_solution():
        covariance = np.diag([0.04, 0.09])
        weights = MeanVarianceSolver(2).solve(np.zeros(2), covariance)
        np.testing.assert_allclose(weights, [9 / 13, 4 / 13], atol=1e-5)
        factor = covariance_factor(covariance)
        np.testing.assert_allclose(factor.T @ factor, covariance)


## 5. 优化层能传递梯度

反向传播产生的均值梯度必须有限且非零。


In [7]:
def test_layer_has_nonzero_finite_gradient():
        mean = torch.tensor([0.03, 0.04], dtype=torch.float64, requires_grad=True)
        factor = torch.tensor(covariance_factor(np.diag([0.04, 0.09])))
        weights, = make_allocation_layer(2)(mean, factor)
        weights[0].backward()
        checks.assertTrue(torch.isfinite(mean.grad).all())
        checks.assertGreater(mean.grad.abs().sum().item(), 1e-4)


## 6. 标准化只使用训练集

拟合后的中心必须等于训练特征的均值，不能混入验证数据。


In [8]:
def test_scaler_uses_training_only():
        samples = make_samples(synthetic_returns(days=350, assets=3))
        train = tuple(samples[key][:5] for key in ("features", "factors", "future"))
        validation = tuple(samples[key][5:] for key in ("features", "factors", "future"))
        fitted = train_model(train, validation, "mse", epochs=1)
        np.testing.assert_allclose(fitted.center, train[0].mean(0))


## 7. 无效收益与第一天回撤

拒绝缺失收益；计算最大回撤时必须包含初始资金，不能漏掉第一天的亏损。


In [9]:
def test_invalid_returns_and_first_day_drawdown():
        frame = synthetic_returns(days=100, assets=3)
        frame.iloc[0, 0] = np.nan
        with checks.assertRaises(ValueError):
            validate_returns(frame)
        metrics = summarize([-0.1, 0.0], [0])
        checks.assertAlmostEqual(metrics["max_drawdown"], -0.1)


## 8. 协方差有效性

拒绝不对称、非半正定和缺失值矩阵，同时允许合法的奇异协方差。


In [10]:
def test_covariance_validation():
        for covariance in ([[1, 2], [0, 1]], [[1, 0], [0, -1]], [[np.nan]]):
            with checks.assertRaises(ValueError):
                covariance_factor(covariance)
        singular = np.ones((3, 3))
        factor = covariance_factor(singular)
        np.testing.assert_allclose(factor.T @ factor, singular, atol=1e-12)


## 9. 换手约束确实生效

构造会触及上限的例子，核对预期权重，并拒绝缺少上一期权重或非法上限的调用。


In [11]:
def test_turnover_constraint_binds():
        previous = np.array([0.5, 0.5])
        covariance = np.eye(2) * 0.01
        mean = np.array([1.0, -1.0])
        solver = MeanVarianceSolver(2, turnover_limit=0.2)
        weights = solver.solve(mean, covariance, previous)
        np.testing.assert_allclose(weights, [0.6, 0.4], atol=1e-5)
        with checks.assertRaises(ValueError):
            solver.solve(mean, covariance)
        for limit in (0, -0.1, 2.1, np.nan):
            with checks.assertRaises(ValueError):
                MeanVarianceSolver(2, turnover_limit=limit)


## 10. 梯度与有限差分一致

把自动微分结果与精确求解器的中心有限差分比较，而非只检查梯度是否存在。


In [12]:
def test_gradient_matches_finite_difference():
        covariance = np.diag([0.04, 0.09])
        mean = torch.tensor([0.03, 0.04], dtype=torch.float64, requires_grad=True)
        factor = torch.tensor(covariance_factor(covariance))
        layer = make_allocation_layer(2)
        weights, = layer(mean, factor, solver_args={"eps": 1e-8})
        weights[0].backward()
        solver = MeanVarianceSolver(2)
        epsilon = 1e-4
        numeric = []
        for direction in np.eye(2) * epsilon:
            upper = solver.solve(mean.detach().numpy() + direction, covariance)[0]
            lower = solver.solve(mean.detach().numpy() - direction, covariance)[0]
            numeric.append((upper - lower) / (2 * epsilon))
        np.testing.assert_allclose(mean.grad.numpy(), numeric, atol=2e-3, rtol=2e-3)


## 11. 带约束的优化层与求解器一致

在相同数据和换手上限下，两种求解接口必须产生一致的权重。


In [13]:
def test_constrained_layer_matches_solver():
        mean = torch.tensor([0.8, -0.2], dtype=torch.float64, requires_grad=True)
        covariance = np.eye(2) * 0.05
        previous = torch.tensor([0.4, 0.6], dtype=torch.float64)
        factor = torch.tensor(covariance_factor(covariance))
        weights, = make_allocation_layer(2, turnover_limit=0.3)(
            mean, factor, previous, solver_args={"eps": 1e-8}
        )
        expected = MeanVarianceSolver(2, turnover_limit=0.3).solve(
            mean.detach().numpy(), covariance, previous.numpy()
        )
        np.testing.assert_allclose(weights.detach().numpy(), expected, atol=2e-4)


## 12. 神经网络的换手限制

检查沿单纯形线段调整后的权重仍然满仓、满足换手上限，并保留有限的梯度。


In [14]:
def test_neural_turnover_projection():
        previous = torch.tensor([[0.2, 0.8], [0.5, 0.5]])
        proposed = torch.tensor([[0.9, 0.1], [0.5, 0.5]], requires_grad=True)
        result = constrain_turnover(proposed, previous, 0.2)
        checks.assertTrue(torch.isfinite(result).all())
        checks.assertTrue(((result - previous).abs().sum(1) <= 0.200001).all())
        np.testing.assert_allclose(result.detach().numpy().sum(1), 1)
        result[:, 0].sum().backward()
        checks.assertTrue(torch.isfinite(proposed.grad).all())


## 13. French 数据解析

构造微型 ZIP 数据，检查解析器选择价值加权表、转换百分比并识别缺失值标记。


In [15]:
def test_archive_parser_selects_value_weighted_table():
        text = ("Average Value Weighted Returns -- Daily\n,A,B\n"
                "20200102,1.0,2.0\n20200103,-99.99,0.0\n\n"
                "Average Equal Weighted Returns -- Daily\n,A,B\n20200102,9,9\n")
        payload = BytesIO()
        with ZipFile(payload, "w") as archive:
            archive.writestr("returns.csv", text)
        frame = parse_french(payload.getvalue())
        checks.assertEqual(frame.shape, (2, 2))
        checks.assertAlmostEqual(frame.iloc[0, 0], 0.01)
        checks.assertTrue(np.isnan(frame.iloc[1, 0]))


## 14. 近似优化器完整流程

运行小规模训练与测试，检查约束、目标差距和模型文件。这里的规模只用于测试。


In [16]:
def test_amortized_end_to_end():
        torch.set_num_threads(1)
        with tempfile.TemporaryDirectory() as output:
            report = run_experiment(output, n_assets=3, train_size=16,
                                    test_size=8, epochs=2, turnover_limit=0.4)
            checks.assertLessEqual(report["maximum_turnover"], 0.40001)
            checks.assertLess(report["maximum_budget_error"], 1e-6)
            checks.assertGreaterEqual(report["objective_gap_mean"], -1e-6)
            checks.assertTrue((Path(output) / "amortized_model.pt").exists())


## 15. 滚动回测完整流程

运行六种策略的小规模滚动回测，检查日期先后关系、输出数据、权重与图片。


In [17]:
def test_walk_forward_end_to_end():
        torch.set_num_threads(1)
        frame = synthetic_returns(days=240, assets=3)
        config = Config(lookback=63, horizon=10, train_periods=5,
                        validation_periods=2, test_periods=3, epochs=1)
        with tempfile.TemporaryDirectory() as output:
            report = run_backtest(frame, output, config)
            daily = pd.read_csv(Path(output) / "daily_returns.csv", index_col=0)
            weights = pd.read_csv(Path(output) / "weights.csv")
            checks.assertEqual(daily.shape[1], 6)
            checks.assertEqual(len(daily), report["test_days"])
            checks.assertTrue(np.isfinite(daily.to_numpy()).all())
            checks.assertFalse(daily.index.duplicated().any())
            np.testing.assert_allclose(weights[list(frame.columns)].sum(axis=1), 1)
            for fold in report["folds"]:
                checks.assertLess(fold["training_last_label"], fold["validation_first_label"])
                checks.assertLess(fold["validation_last_label"], fold["test_start"])
            checks.assertTrue((Path(output) / "backtest.png").exists())


## 执行全部测试

最后这个单元格真正运行上面定义的 15 项检查。完整日志同时保存到 `docs/test_results.txt`；任何失败都会使 Notebook 报错。


In [18]:
test_functions = [
    test_future_changes_do_not_change_current_features,
    test_drift_cost_and_initial_cash,
    test_differentiable_holding_matches_accounting,
    test_minimum_variance_matches_analytic_solution,
    test_layer_has_nonzero_finite_gradient,
    test_scaler_uses_training_only,
    test_invalid_returns_and_first_day_drawdown,
    test_covariance_validation,
    test_turnover_constraint_binds,
    test_gradient_matches_finite_difference,
    test_constrained_layer_matches_solver,
    test_neural_turnover_projection,
    test_archive_parser_selects_value_weighted_table,
    test_amortized_end_to_end,
    test_walk_forward_end_to_end,
]


In [19]:
suite = unittest.TestSuite(unittest.FunctionTestCase(test) for test in test_functions)
log = StringIO()
with redirect_stdout(log), redirect_stderr(log):
    result = unittest.TextTestRunner(stream=log, verbosity=2).run(suite)

Path("docs").mkdir(exist_ok=True)
Path("docs/test_results.txt").write_text(log.getvalue())
print(log.getvalue())
assert result.testsRun == 15, "测试数应为 15。"
assert result.wasSuccessful(), "测试未全部通过，请查看上面的错误信息。"
print(f"全部 {result.testsRun} 项测试通过。")


unittest.case.FunctionTestCase (test_future_changes_do_not_change_current_features) ... ok
unittest.case.FunctionTestCase (test_drift_cost_and_initial_cash) ... ok
unittest.case.FunctionTestCase (test_differentiable_holding_matches_accounting) ... ok
unittest.case.FunctionTestCase (test_minimum_variance_matches_analytic_solution) ... ok
unittest.case.FunctionTestCase (test_layer_has_nonzero_finite_gradient) ... ok
unittest.case.FunctionTestCase (test_scaler_uses_training_only) ... ok
unittest.case.FunctionTestCase (test_invalid_returns_and_first_day_drawdown) ... ok
unittest.case.FunctionTestCase (test_covariance_validation) ... ok
unittest.case.FunctionTestCase (test_turnover_constraint_binds) ... ok
unittest.case.FunctionTestCase (test_gradient_matches_finite_difference) ... ok
unittest.case.FunctionTestCase (test_constrained_layer_matches_solver) ... ok
unittest.case.FunctionTestCase (test_neural_turnover_projection) ... ok
unittest.case.FunctionTestCase (test_archive_parser_selects